# PPI Time Alignment (Plotly Version)

Інтерактивна версія для zoom/pan по всьому часовому діапазону.
Джерело істини для canonical графіка: `normalized parquet + time_alignment_report.json`.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import os

SESSION_ID = "a3b4f7a4-c810-4e7a-8152-e9a05dd32b1e"
MAX_POINTS = 120_000

if os.environ.get("PLOTLY_RENDERER"):
    pio.renderers.default = os.environ["PLOTLY_RENDERER"]


In [ ]:
def resolve_cached_session_root(session_id: str) -> Path:
    cands = [
        Path("notebooks/data_cache") / f"session_id={session_id}",
        Path("notebooks/notebooks/data_cache") / f"session_id={session_id}",
    ]
    for c in cands:
        if c.exists():
            return c
    raise FileNotFoundError(f"No local cached session root found for {session_id}. Checked: {cands}")


def resolve_session_paths(session_id: str):
    root = resolve_cached_session_root(session_id)
    raw_jsonl = next(root.glob("data/wearable/raw/**/session_id=*/streams/ppi/chunks.jsonl"))
    clean_parquet = next(root.glob("data/wearable/processed/clean_timeseries/**/session_id=*/streams/ppi/data.parquet"))
    report_json = next(root.glob("data/wearable/processed/clean_timeseries/**/session_id=*/streams/ppi/time_alignment_report.json"))
    return root, raw_jsonl, clean_parquet, report_json


root, RAW_JSONL_PATH, CLEAN_PARQUET_PATH, REPORT_JSON_PATH = resolve_session_paths(SESSION_ID)
print("root:", root)
print("raw:", RAW_JSONL_PATH)
print("clean:", CLEAN_PARQUET_PATH)
print("report:", REPORT_JSON_PATH)


In [ ]:
def load_raw_ppi(path: Path) -> tuple[pd.DataFrame, list[dict]]:
    chunks = []
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            chunks.append(obj)
            payload = obj.get("payload") or {}
            samples = payload.get("samples") or []
            for sample_idx, s in enumerate(samples):
                row = {
                    "line_no": line_no,
                    "sample_idx_in_chunk": sample_idx,
                    "sequence": obj.get("sequence"),
                    "session_id": obj.get("session_id"),
                    "stream_id": obj.get("stream_id"),
                }
                row.update(s)
                rows.append(row)
    df = pd.DataFrame(rows).reset_index(drop=True)
    df["row_idx"] = np.arange(len(df))
    for col in ["timeStamp", "ppInMs", "ppErrorEstimate", "blockerBit", "skinContactStatus", "skinContactSupported"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df, chunks


def add_raw_quality(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["pp_s"] = out["ppInMs"].clip(lower=0).fillna(0) / 1000.0
    out["valid_contact"] = out.get("skinContactStatus", 1).eq(1)
    out["not_blocked"] = out.get("blockerBit", 0).eq(0)
    out["pp_error_ok_strict"] = out.get("ppErrorEstimate", 999).le(10)
    out["pp_error_ok_med"] = out.get("ppErrorEstimate", 999).le(30)
    out["physio_ok"] = out["ppInMs"].between(300, 2200)

    out["quality_tier"] = np.select(
        [
            out["not_blocked"] & out["valid_contact"] & out["pp_error_ok_strict"] & out["physio_ok"],
            out["not_blocked"] & out["valid_contact"] & out["pp_error_ok_med"] & out["physio_ok"],
        ],
        ["high", "medium"],
        default="low",
    )
    return out


def build_reconstructed_timeline(df: pd.DataFrame, startup_delay_s: float = 0.0) -> pd.DataFrame:
    out = df.copy().reset_index(drop=True)
    out["pp_s"] = out["ppInMs"].clip(lower=0).fillna(0) / 1000.0
    out["t_reconstructed_s"] = out["pp_s"].shift(fill_value=0).cumsum() + float(startup_delay_s)
    return out


def load_normalized(clean_path: Path, report_path: Path):
    dfn = pd.read_parquet(clean_path).copy()
    dfn["ts_utc"] = pd.to_datetime(dfn["ts_utc"], utc=True, errors="coerce")
    rpt = json.loads(report_path.read_text(encoding="utf-8"))
    start = pd.to_datetime(rpt.get("session_window_start"), utc=True, errors="coerce")
    end = pd.to_datetime(rpt.get("session_window_end"), utc=True, errors="coerce")
    return dfn, rpt, start, end


raw_df, raw_chunks = load_raw_ppi(RAW_JSONL_PATH)
raw_df = add_raw_quality(raw_df)
norm_df, report, norm_start, norm_end = load_normalized(CLEAN_PARQUET_PATH, REPORT_JSON_PATH)

print("raw rows:", len(raw_df), "normalized rows:", len(norm_df))
print("basis:", report.get("alignment_basis"), "level:", report.get("alignment_basis_level"))
print("window:", norm_start, "->", norm_end)


In [ ]:
def _downsample(df: pd.DataFrame, max_points: int = MAX_POINTS) -> pd.DataFrame:
    if len(df) <= max_points:
        return df
    step = max(1, len(df) // max_points)
    return df.iloc[::step].copy()


def plot_raw_reconstructed_plotly(raw_df: pd.DataFrame, chunks: list[dict], startup_delay_s: float = 0.0, max_points: int = MAX_POINTS):
    # anchor from raw metadata (recording_start preferred)
    t = (chunks[0].get("time") or {}) if chunks else {}
    anchor_start = pd.to_datetime(t.get("recording_start_utc") or t.get("fetch_started_at_collector") or t.get("first_sample_received_at_collector"), utc=True, errors="coerce")
    if pd.isna(anchor_start):
        raise RuntimeError("No usable start anchor in raw metadata")

    recon = build_reconstructed_timeline(raw_df, startup_delay_s=startup_delay_s)
    recon["t_abs_utc"] = anchor_start + pd.to_timedelta(recon["t_reconstructed_s"], unit="s")
    reconstructed_end = recon["t_abs_utc"].iloc[-1]

    plot_df = _downsample(recon, max_points=max_points)

    fig = px.scatter(
        plot_df,
        x="t_abs_utc",
        y="ppInMs",
        color="quality_tier",
        color_discrete_map={"high":"green","medium":"orange","low":"red"},
        title=f"RAW->Reconstructed PPI (Plotly) | delay={startup_delay_s}s",
        opacity=0.7,
        hover_data=["row_idx","ppErrorEstimate","blockerBit","skinContactStatus"],
    )

    fig.add_vline(x=anchor_start.to_pydatetime(), line_dash="dash", line_color="green")
    fig.add_vline(x=reconstructed_end.to_pydatetime(), line_dash="dash", line_color="red")

    fig.update_layout(height=520)
    fig.update_xaxes(rangeslider_visible=True)
    fig.show()

    print("anchor_start:", anchor_start)
    print("reconstructed_end:", reconstructed_end)
    print("duration_s:", float((reconstructed_end - anchor_start).total_seconds()))


plot_raw_reconstructed_plotly(raw_df, raw_chunks, startup_delay_s=0.0)


In [ ]:
def plot_normalized_canonical_plotly(dfn: pd.DataFrame, report: dict, start: pd.Timestamp, end: pd.Timestamp, max_points: int = MAX_POINTS):
    basis = str(report.get("alignment_basis"))
    level = str(report.get("alignment_basis_level"))
    is_new = (basis == "ppi_cumulative_reconstruction" and level == "L2")

    if not is_new:
        raise RuntimeError("Normalized cache is not L2 cumulative. Refresh local cache from server before plotting canonical chart.")

    x = dfn.sort_values("ts_utc").copy()
    x = _downsample(x, max_points=max_points)

    y_col = "pp_in_ms" if "pp_in_ms" in x.columns else ("ppInMs" if "ppInMs" in x.columns else None)
    if y_col is None:
        raise RuntimeError("No pp column found in normalized dataframe")

    color_col = "sample_quality_tier" if "sample_quality_tier" in x.columns else None

    fig = px.scatter(
        x,
        x="ts_utc",
        y=y_col,
        color=color_col,
        color_discrete_map={"high":"green","medium":"orange","low":"red"},
        title="Normalized PPI vs chosen session anchors (Plotly canonical)",
        opacity=0.65,
        hover_data=[c for c in ["sample_index","timestamp_origin","pp_error_estimate","blocker_bit"] if c in x.columns],
    )

    fig.add_vline(x=start.to_pydatetime(), line_dash="dash", line_color="green")
    fig.add_vline(x=end.to_pydatetime(), line_dash="dash", line_color="red")

    pad = (end - start) * 0.02
    fig.update_xaxes(range=[start - pad, end + pad], rangeslider_visible=True)
    fig.update_layout(height=520)
    fig.show()

    inside = ((x["ts_utc"] >= start) & (x["ts_utc"] <= end)).mean()
    print("inside_ratio:", float(inside))
    print("ts_min/max:", x["ts_utc"].min(), x["ts_utc"].max())


plot_normalized_canonical_plotly(norm_df, report, norm_start, norm_end)
